# Template
Copy this notebook to start a new lesson. Edit only the config cell to scale up on the 5090.

In [ ]:
# --- Config: bump these on the 5090 (e.g. BS=128, IMG_SIZE=448) ---
BS = 32          # fits 8 GB on the 3070 Ti
IMG_SIZE = 224
FP16 = True      # use learn.to_fp16() when True

In [ ]:
import os
from pathlib import Path

# Project root = parent of nbs/. Keep fastai downloads inside the git-ignored data/ folder.
PROJECT = Path.cwd().resolve().parent if Path.cwd().name == 'nbs' else Path.cwd().resolve()
DATA = Path(os.environ.get('LOCAL_AI_DATA', PROJECT / 'data'))
MODELS = Path(os.environ.get('LOCAL_AI_MODELS', PROJECT / 'models'))
os.environ.setdefault('FASTAI_HOME', str(DATA / 'fastai'))  # must be set before importing fastai

from fastai.vision.all import *
import torch
print(torch.cuda.get_device_name(0), '|', 'BS', BS, '| IMG_SIZE', IMG_SIZE, '| FP16', FP16)

In [ ]:
from ddgs import DDGS #DuckDuckGo has changed the api so we need to update 
from fastcore.all import *

def search_images(keywords, max_images=200): return L(DDGS().images(keywords, max_results=max_images)).itemgot('image')
import time, json

In [ ]:
#NB: `search_images` depends on duckduckgo.com, which doesn't always return correct responses.
#    If you get a JSON error, just try running it again (it may take a couple of tries).
urls = search_images('bird photos', max_images=1)
urls[0]

In [ ]:
from fastdownload import download_url
dest = 'bird.jpg'
download_url(urls[0], dest, show_progress=False)

from fastai.vision.all import *
im = Image.open(dest)
im.to_thumb(256,256)

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

# Plain ImageNet ResNet18, no fine-tuning. Weights go to data/torch instead of ~/.cache.
os.environ.setdefault('TORCH_HOME', str(DATA / 'torch'))
weights = ResNet18_Weights.IMAGENET1K_V1
model = resnet18(weights=weights).eval().cuda()
preprocess = weights.transforms()          # the resize/normalize ResNet18 was trained with
categories = weights.meta['categories']

# ImageNet classes that are birds: cock..great grey owl, black grouse..black swan, white stork..albatross
BIRD_IDXS = [*range(7, 25), *range(80, 101), *range(127, 147)]

def bird_or_not(fname):
    x = preprocess(PILImage.create(fname)).unsqueeze(0).cuda()
    with torch.no_grad(): probs = model(x).softmax(dim=1)[0]
    p_bird = probs[BIRD_IDXS].sum().item()
    top = probs.argmax().item()
    print(f"{'bird' if p_bird > 0.5 else 'not bird'} | P(bird)={p_bird:.4f} | top ImageNet class: {categories[top]} ({probs[top]:.3f})")

bird_or_not('small_bird.jpg')

In [ ]:
download_url(search_images('forest with far away bird', max_images=1)[0], 'small_bird.jpg', show_progress=False)
Image.open('small_bird.jpg').to_thumb(256,256)

In [ ]:
searches = 'animal in forest','bird'
path = Path('bird_or_not')

for o in searches:
    dest = (path/o)
    dest.mkdir(exist_ok=True, parents=True)
    download_images(dest, urls=search_images(f'{o} photo'))
    time.sleep(5)
    resize_images(path/o, max_size=400, dest=path/o)

In [ ]:
failed = verify_images(get_image_files(path))
failed.map(Path.unlink)
len(failed)

In [ ]:
dls = DataBlock(
    blocks=(ImageBlock, CategoryBlock), 
    get_items=partial(get_image_files, folders=['bird', 'animal in forest']), 
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    get_y=parent_label,
    item_tfms=[Resize(192, method='squish')]
).dataloaders(path, bs=32)

dls.show_batch(max_n=6)


In [ ]:
learn = vision_learner(dls, resnet18, metrics=error_rate)
learn.fine_tune(3)

In [ ]:
bird_idx = learn.dls.vocab.o2i['bird']
pred,_,probs = learn.predict(PILImage.create('small_bird.jpg'))
print(f"This is a: {pred}.")
print(f"Probability it's a bird: {probs[bird_idx]:.4f}")